# 02 — Data Cleaning

**Input:** `data/interim/groundwater_merged_2018_2020.csv` (from `01_Merge_Raw`)

Now that the rows are stacked, make the *values* trustworthy. Steps:

1. **Coerce chemistry to numeric** — blanks / stray text become `NaN`.
2. **Normalise text** — trim + case-standardise district/mandal/village and the
   classification strings (which arrive as `postmonsoon 2018 `, `post monsoon 2019`, ...).
3. **Normalise `season`** to a single `post_monsoon` value.
4. **Handle missing values**
   - chemistry: **left as `NaN` by design** (imputing invented chemistry would
     bias downstream risk labels)
   - `gwl` (groundwater level): imputed by **district median**, then global median
     — spatial imputation is defensible for a physical water-table depth.
5. **Drop exact duplicate rows.**

No modelling labels or feature engineering here — that is `03_Data_Preprocessing`.

**Output:** `data/processed/groundwater_clean_2018_2020.csv`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM = ROOT / "data" / "interim" / "groundwater_merged_2018_2020.csv"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INTERIM)
print("Loaded:", df.shape)
df.head()

Loaded: (1106, 27)


,sno,district,mandal,village,lat,lon,gwl,season,ph,ec,...,na,k,ca,mg,th,sar,irrigation_class,rsc,rsc_class,year
0,1,ADILABAD,Adilabad,Adilabad,19.668300,78.524700,5.09,postmonsoon 2018,8.28,745,...,49.0,4.0,48.0,38.896,279.934211,1.273328,C2S1,-1.198684,P.S.,2018
1,2,ADILABAD,Bazarhatnur,Bazarhatnur,19.458888,78.350833,5.10,postmonsoon 2018,8.29,921,...,42.0,5.0,56.0,63.206,399.893092,0.913166,C3S1,-3.397862,P.S.,2018
2,3,ADILABAD,Gudihatnoor,Gudihatnoor,19.525555,78.512222,4.98,postmonsoon 2018,7.69,510,...,45.0,2.0,24.0,38.896,219.934211,1.319284,C2S1,-0.398684,P.S.,2018
3,4,ADILABAD,Jainath,Jainath,19.730555,78.640000,5.75,postmonsoon 2018,8.09,422,...,27.0,1.0,32.0,19.448,159.967105,0.928155,C2S1,0.000658,P.S.,2018
4,5,ADILABAD,Narnoor,Narnoor,19.495665,78.852654,2.15,postmonsoon 2018,8.21,2321,...,298.0,5.0,56.0,92.378,519.843750,5.682664,C4S2,-4.396875,P.S.,2018


## 1. Coerce chemistry columns to numeric

Anything non-numeric (stray text, blanks) becomes `NaN` rather than silently breaking later math.

In [2]:
NUMERIC_COLS = ["lat","lon","gwl","ph","ec","tds","co3","hco3","cl","f",
                "no3","so4","na","k","ca","mg","th","sar","rsc"]

before_na = df[NUMERIC_COLS].isna().sum().sum()
for c in NUMERIC_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
after_na = df[NUMERIC_COLS].isna().sum().sum()
print(f"NaNs in numeric block: {before_na} -> {after_na} (coercion exposed {after_na-before_na})")
df[NUMERIC_COLS].dtypes

NaNs in numeric block: 171 -> 172 (coercion exposed 1)


lat     float64
lon     float64
gwl     float64
ph      float64
ec        int64
tds     float64
co3     float64
hco3    float64
cl        int64
f       float64
no3     float64
so4     float64
na      float64
k       float64
ca      float64
mg      float64
th      float64
sar     float64
rsc     float64
dtype: object

## 2. Normalise text columns

In [3]:
for c in ["district", "mandal", "village"]:
    df[c] = df[c].astype(str).str.strip().str.title()
for c in ["irrigation_class", "rsc_class"]:
    df[c] = df[c].astype(str).str.strip().str.upper()

print("Districts:", df["district"].nunique())
print(sorted(df["district"].unique())[:10], "...")

Districts: 33
['Adilabad', 'Bhadradri', 'Bhupalpally', 'Hyderabad', 'Jagityal', 'Jangaon', 'Jogulamba(Gadwal)', 'Kamareddy', 'Karimnagar', 'Khammam'] ...


## 3. Normalise `season`

The three files wrote this three different ways; all rows are the post-monsoon sampling round.

In [4]:
print("Raw season values:", df["season"].unique())
df["season"] = "post_monsoon"
print("Normalised ->", df["season"].unique())

Raw season values: <StringArray>
['postmonsoon 2018 ', 'post monsoon 2019', 'Post-monsoon 2020']
Length: 3, dtype: str
Normalised -> <StringArray>
['post_monsoon']
Length: 1, dtype: str


## 4. Missing values

**Chemistry stays `NaN`.** Only `gwl` is imputed, spatially (district median → global median).

In [5]:
print("Missing gwl before:", int(df["gwl"].isna().sum()))
district_median = df.groupby("district")["gwl"].transform("median")
df["gwl"] = df["gwl"].fillna(district_median)
df["gwl"] = df["gwl"].fillna(df["gwl"].median())
print("Missing gwl after :", int(df["gwl"].isna().sum()))

chem = [c for c in NUMERIC_COLS if c not in ("lat","lon","gwl")]
remaining = df[chem].isna().sum()
remaining = remaining[remaining > 0]
print("\nChemistry NaNs left intact by design:")
print(remaining.to_string() if len(remaining) else "  (none)")

Missing gwl before: 11
Missing gwl after : 0

Chemistry NaNs left intact by design:
ph       1
co3    160


## 5. Drop exact duplicates

In [6]:
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows: {n0} -> {len(df)} ({n0-len(df)} exact duplicates removed)")

Rows: 1106 -> 1106 (0 exact duplicates removed)


## Reorder & save

Identifiers → location → chemistry → indices. No derived labels yet.

In [7]:
ordered = (
    ["year","season","sno","district","mandal","village","lat","lon","gwl"]
    + ["ph","ec","tds","co3","hco3","cl","f","no3","so4","na","k","ca","mg","th"]
    + ["sar","rsc","irrigation_class","rsc_class"]
)
ordered = [c for c in ordered if c in df.columns]
df = df[ordered]

out = OUT_DIR / "groundwater_clean_2018_2020.csv"
df.to_csv(out, index=False)
print("Wrote", out.relative_to(ROOT), "|", df.shape)
df.head()

Wrote data/processed/groundwater_clean_2018_2020.csv | (1106, 27)


,year,season,sno,district,mandal,village,lat,lon,gwl,ph,...,so4,na,k,ca,mg,th,sar,rsc,irrigation_class,rsc_class
0,2018,post_monsoon,1,Adilabad,Adilabad,Adilabad,19.668300,78.524700,5.09,8.28,...,46.0,49.0,4.0,48.0,38.896,279.934211,1.273328,-1.198684,C2S1,P.S.
1,2018,post_monsoon,2,Adilabad,Bazarhatnur,Bazarhatnur,19.458888,78.350833,5.10,8.29,...,68.0,42.0,5.0,56.0,63.206,399.893092,0.913166,-3.397862,C3S1,P.S.
2,2018,post_monsoon,3,Adilabad,Gudihatnoor,Gudihatnoor,19.525555,78.512222,4.98,7.69,...,44.0,45.0,2.0,24.0,38.896,219.934211,1.319284,-0.398684,C2S1,P.S.
3,2018,post_monsoon,4,Adilabad,Jainath,Jainath,19.730555,78.640000,5.75,8.09,...,35.0,27.0,1.0,32.0,19.448,159.967105,0.928155,0.000658,C2S1,P.S.
4,2018,post_monsoon,5,Adilabad,Narnoor,Narnoor,19.495665,78.852654,2.15,8.21,...,280.0,298.0,5.0,56.0,92.378,519.843750,5.682664,-4.396875,C4S2,P.S.
